In [7]:
%pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [8]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

# Fetch the official UCI dataset
steel = fetch_ucirepo(id=198)

# Separate input features and target columns
X = steel.data.features.copy()
Y = steel.data.targets.copy()

# Display dimensions
print("INPUT FEATURES SHAPE:", X.shape)
print("TARGET SHAPE:", Y.shape)
print("TOTAL RECORDS:", len(X))

# Display feature names
print("\n27 INPUT FEATURES:")
print(X.columns.tolist())

# Display target names
print("\n7 TARGET COLUMNS:")
print(Y.columns.tolist())

# Check missing values
print("\nMISSING VALUES IN FEATURES:", X.isnull().sum().sum())
print("MISSING VALUES IN TARGETS:", Y.isnull().sum().sum())

# Check binary target encoding
print("\nTARGET VALUE COUNTS:")
for col in Y.columns:
    print(col, Y[col].value_counts().to_dict())

# Check whether every row has exactly one active target
row_target_sum = Y.sum(axis=1)

print("\nTARGET FLAGS PER ROW:")
print(row_target_sum.value_counts().sort_index())

print("\nROWS WITH ZERO ACTIVE TARGETS:",
      (row_target_sum == 0).sum())

print("ROWS WITH MULTIPLE ACTIVE TARGETS:",
      (row_target_sum > 1).sum())

# Class distribution
assert Y.isin([0, 1]).all().all(), "Non-binary target values found"
assert (row_target_sum == 1).all(), "Target encoding is not exactly one-hot"

class_labels = Y.idxmax(axis=1)
class_distribution = class_labels.value_counts()

print("\nFINAL CLASS DISTRIBUTION:")
print(class_distribution)

print("\nCLASS PERCENTAGES:")
print((class_distribution / len(Y) * 100).round(2))

# Final checks
assert X.shape == (1941, 27)
assert Y.shape == (1941, 7)
assert X.isnull().sum().sum() == 0
assert Y.isnull().sum().sum() == 0

print("\nSTAGE 1 DATASET CHECKS PASSED")

INPUT FEATURES SHAPE: (1941, 27)
TARGET SHAPE: (1941, 7)
TOTAL RECORDS: 1941

27 INPUT FEATURES:
['X_Minimum', 'X_Maximum', 'Y_Minimum', 'Y_Maximum', 'Pixels_Areas', 'X_Perimeter', 'Y_Perimeter', 'Sum_of_Luminosity', 'Maximum_of_Luminosity', 'Length_of_Conveyer', 'TypeOfSteel_A300', 'TypeOfSteel_A400', 'Steel_Plate_Thickness', 'Edges_Index', 'Empty_Index', 'Square_Index', 'Outside_X_Index', 'Edges_X_Index', 'Edges_Y_Index', 'Outside_Global_Index', 'LogOfAreas', 'Log_X_Index', 'Log_Y_Index', 'Orientation_Index', 'Luminosity_Index', 'SigmoidOfAreas', 'Minimum_of_Luminosity']

7 TARGET COLUMNS:
['Pastry', 'Z_Scratch', 'K_Scratch', 'Stains', 'Dirtiness', 'Bumps', 'Other_Faults']

MISSING VALUES IN FEATURES: 0
MISSING VALUES IN TARGETS: 0

TARGET VALUE COUNTS:
Pastry {0: 1783, 1: 158}
Z_Scratch {0: 1751, 1: 190}
K_Scratch {0: 1550, 1: 391}
Stains {0: 1869, 1: 72}
Dirtiness {0: 1886, 1: 55}
Bumps {0: 1539, 1: 402}
Other_Faults {0: 1268, 1: 673}

TARGET FLAGS PER ROW:
1    1941
Name: count, d

In [23]:
import pandas as pd
from pathlib import Path
from ucimlrepo import fetch_ucirepo

# Create data folder
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# Fetch official UCI dataset
steel = fetch_ucirepo(id=198)

X = steel.data.features.copy()
Y = steel.data.targets.copy()

# Save original dataset with all 34 columns
raw_data = pd.concat([X, Y], axis=1)

raw_path = DATA_DIR / "steel_plates_raw.csv"
raw_data.to_csv(raw_path, index=False)

print("Original dataset saved successfully!")
print("Location:", raw_path.resolve())
print("Shape:", raw_data.shape)

Original dataset saved successfully!
Location: /home/s2026ak05264/workspace/ML_Assignement_2/notebooks/data/steel_plates_raw.csv
Shape: (1941, 34)


In [22]:
# Convert seven binary columns into one target column
y = Y.idxmax(axis=1)

# Prepare final classification dataset
df = X.copy()
df["Target"] = y

print("Final dataset shape:", df.shape)

print("\nTarget column:")
print(df["Target"].head())

print("\nClass distribution:")
print(df["Target"].value_counts())

Final dataset shape: (1941, 28)

Target column:
0    Pastry
1    Pastry
2    Pastry
3    Pastry
4    Pastry
Name: Target, dtype: str

Class distribution:
Target
Other_Faults    673
Bumps           402
K_Scratch       391
Z_Scratch       190
Pastry          158
Stains           72
Dirtiness        55
Name: count, dtype: int64


In [11]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [12]:
import numpy as np
import scipy
import sklearn

print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)

NumPy: 2.5.4
SciPy: 1.18.1
Scikit-learn: 1.9.1


In [13]:
import os

print(os.getcwd())

/home/s2026ak05264/workspace/ML_Assignement_2/notebooks


In [14]:
print(os.listdir())

['02_model_training_evaluation.ipynb', 'data', '.ipynb_checkpoints', '01_dataset_inspection.ipynb']


In [15]:
import pandas as pd

df = pd.read_csv("data/steel_plates_raw.csv")

print(df.head())
print(df.shape)

   X_Minimum  X_Maximum  Y_Minimum  Y_Maximum  Pixels_Areas  X_Perimeter  \
0         42         50     270900     270944           267           17   
1        645        651    2538079    2538108           108           10   
2        829        835    1553913    1553931            71            8   
3        853        860     369370     369415           176           13   
4       1289       1306     498078     498335          2409           60   

   Y_Perimeter  Sum_of_Luminosity  Maximum_of_Luminosity  Length_of_Conveyer  \
0           44              24220                    108                1687   
1           30              11397                    123                1687   
2           19               7972                    125                1623   
3           45              18996                    126                1353   
4          260             246930                    126                1353   

   ...  Luminosity_Index  SigmoidOfAreas  Minimum_of_Luminosit

In [24]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop(columns=["Target"])
y = df["Target"]

# Stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Training features:", X_train.shape[1])
print("Testing features:", X_test.shape[1])

Training records: 1552
Testing records: 389
Training features: 27
Testing features: 27


In [18]:
import sys

!{sys.executable} -m pip install --upgrade numpy scipy scikit-learn

In [19]:
# Combine features and target
train_data = X_train.copy()
train_data["Target"] = y_train

test_data = X_test.copy()
test_data["Target"] = y_test

# Save CSV files
train_data.to_csv(DATA_DIR / "train_data.csv", index=False)
test_data.to_csv(DATA_DIR / "test_data.csv", index=False)

print("Training and testing datasets saved successfully!")

print("\nSaved files:")
for file in DATA_DIR.glob("*.csv"):
    print(file.name, "->", file.stat().st_size, "bytes")

Training and testing datasets saved successfully!

Saved files:
steel_plates_raw.csv -> 305387 bytes
steel_plates_full.csv -> 297288 bytes
train_data.csv -> 237843 bytes
test_data.csv -> 59853 bytes


In [20]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop(columns=["Target"])
y = df["Target"]

# Stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Training features:", X_train.shape[1])
print("Testing features:", X_test.shape[1])

Training records: 1552
Testing records: 389
Training features: 27
Testing features: 27


In [21]:
# Combine features and target
train_data = X_train.copy()
train_data["Target"] = y_train

test_data = X_test.copy()
test_data["Target"] = y_test

# Save CSV files
train_data.to_csv(DATA_DIR / "train_data.csv", index=False)
test_data.to_csv(DATA_DIR / "test_data.csv", index=False)

print("Training and testing datasets saved successfully!")

print("\nSaved files:")
for file in DATA_DIR.glob("*.csv"):
    print(file.name, "->", file.stat().st_size, "bytes")

Training and testing datasets saved successfully!

Saved files:
steel_plates_raw.csv -> 305387 bytes
steel_plates_full.csv -> 297288 bytes
train_data.csv -> 237843 bytes
test_data.csv -> 59853 bytes


In [22]:
# Verify dimensions
assert train_data.shape == (1552, 28)
assert test_data.shape == (389, 28)

# Verify target categories
assert set(train_data["Target"].unique()) == set(y.unique())
assert set(test_data["Target"].unique()) == set(y.unique())

# Verify missing values
assert train_data.isnull().sum().sum() == 0
assert test_data.isnull().sum().sum() == 0

# Verify feature columns match
assert list(train_data.columns) == list(test_data.columns)

print("STAGE 2 VALIDATION PASSED")

print("\nTraining distribution:")
print(train_data["Target"].value_counts().sort_index())

print("\nTesting distribution:")
print(test_data["Target"].value_counts().sort_index())

STAGE 2 VALIDATION PASSED

Training distribution:
Target
Bumps           321
Dirtiness        44
K_Scratch       313
Other_Faults    538
Pastry          126
Stains           58
Z_Scratch       152
Name: count, dtype: int64

Testing distribution:
Target
Bumps            81
Dirtiness        11
K_Scratch        78
Other_Faults    135
Pastry           32
Stains           14
Z_Scratch        38
Name: count, dtype: int64
